# Przewidywanie goli w kolejnych M minutach za pomocą LSTM

Ten notebook uczy prostą, przejrzystą sieć **PyTorch LSTM**, która odpowiada na pytanie:

> **Na podstawie ostatnich `L` minut meczu, jakie jest prawdopodobieństwo, że drużyna strzeli gola w kolejnych `M` minutach `(t, t + M]`?**

### Dlaczego właśnie te dane wejściowe?
Moduł przygotowania danych `src/lstm_data.py` buduje **20 zwartych, przyczynowych cech** na każdą minutę z trzech tabel (`events`, `team_stats`, `metadata`), zostawiając tylko sygnały najlepiej skorelowane z przyszłymi golami i pomijając zbędny szum:
1. **Aktywność z minuty `t` (10 cech)**: kontakty z piłką w polu karnym (`box_touches`, `opp_box_touches`), wejścia w pole karne, kontakty i celne podania w tercji ataku, strzały z pola karnego, geometryczny wskaźnik jakości strzałów (`xg_proxy`), podania kluczowe, wysokie odbiory oraz straty rywala na własnej połowie.
2. **Narastająca przewaga od początku meczu do minuty `t` (6 cech)**: tempo kontaktów w polu karnym na minutę, różnica tempa wejść w pole karne, skumulowany udział posiadania piłki (`poss_share_cum` z `team_stats`), przewaga podań w tercji ataku (`field_tilt_cum`), różnica średniej wysokości gry (`avg_x_diff_cum`) oraz różnica wybić na minutę.
3. **Aktualny stan meczu i czas (4 cechy)**: bieżący bilans bramek `score_diff` (z samobójami przypisanymi drużynie zyskującej), przewaga liczebna po czerwonych kartkach (`man_advantage`), znormalizowana minuta (`t / 90`) oraz `is_home`.

### Wbudowane zabezpieczenia przed wyciekiem danych
- **Granica połowy**: sekwencje i etykiety dla kolejnych `M` minut nigdy nie przechodzą między 1. a 2. połową.
- **Podział całymi meczami**: cały mecz trafia do `train`, `val` albo `test` — sąsiednie minuty tego samego meczu nigdy nie mieszają się między zbiorami.
- **Skalowanie tylko na treningu**: średnia i odchylenie standardowe są liczone wyłącznie na meczach treningowych.

> **Uwaga o PyTorch**: Przed uruchomieniem poniższych komórek upewnij się, że w środowisku `.venv` masz zainstalowany pakiet `torch` (zobacz `requirements.txt`, np. `pip install torch` lub polecenie z CUDA).

## Krok 1 — Konfiguracja i importy

Zmieniamy tylko hiperparametry `HORIZON_MINUTES` (`M`) albo `SEQUENCE_LENGTH` (`L`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import rankdata
from torch.utils.data import DataLoader, TensorDataset

# Ensure the repository root is on sys.path whether Jupyter starts in notebooks/ or root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.lstm_data import FEATURE_DESCRIPTIONS, FEATURE_NAMES, prepare_lstm_dataset

# --- Experiment hyperparameters ---
HORIZON_MINUTES = 10      # M: predict whether a team scores in (t, t + M]
SEQUENCE_LENGTH = 10      # L: number of consecutive past minutes fed to the LSTM
USE_ALL_MATCHES = True    # True = load all per-match CSVs in output/; False = output/combined/

BATCH_SIZE = 128
HIDDEN_SIZE = 32          # Small hidden state keeps overfitting low on ~350 matches
NUM_LAYERS = 1
DROPOUT = 0.25
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 20
PATIENCE = 5              # Early stopping patience on validation ROC-AUC
SEED = 42

DATA_DIR = PROJECT_ROOT / "output"
MODEL_PATH = DATA_DIR / "lstm_goal_predictor.pt"

# Fix random seeds for reproducible splits and weight initialization.
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Using device: {DEVICE} | PyTorch: {torch.__version__}")

## Krok 2 — Przygotowanie sekwencji i przegląd 20 cech

`prepare_lstm_dataset()` wczytuje tabele przez DuckDB, tworzy ciągłą siatkę minutową w każdej połowie, wylicza 20 cech przyczynowych, wycina okna o kształcie `(SEQUENCE_LENGTH, 20)`, dzieli całe mecze na `train / val / test` i standaryzuje dane wyłącznie na statystykach zbioru `train`.

In [ ]:
bundle = prepare_lstm_dataset(
    data_dir=DATA_DIR,
    sequence_length=SEQUENCE_LENGTH,
    horizon_minutes=HORIZON_MINUTES,
    use_all_matches=USE_ALL_MATCHES,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=SEED,
)

print(f"Horizon M = {bundle.horizon_minutes} min | Sequence length L = {bundle.sequence_length} min")
print(f"Number of features per minute: {len(bundle.feature_names)}\n")

for split_name, split in [("train", bundle.train), ("val", bundle.val), ("test", bundle.test)]:
    n_matches = split.meta["match_id"].nunique()
    pos_rate = 100.0 * float(split.y.mean())
    print(f"{split_name:<5} | matches: {n_matches:>3} | samples: {len(split.y):>6,} | "
          f"shape: {split.x.shape} | goal rate: {pos_rate:.2f}%")

# Print the plain-English glossary of all 20 features.
glossary = pd.DataFrame(
    [(i + 1, name, FEATURE_DESCRIPTIONS[name]) for i, name in enumerate(FEATURE_NAMES)],
    columns=["#", "Feature", "Meaning"],
)
display(glossary)

In [ ]:
# Inspect one concrete sequence (unscaled values reconstructed from scaler mean/std) to verify it makes sense.
sample_idx = 0
sample_meta = bundle.train.meta.iloc[sample_idx]
sample_raw = bundle.train.x[sample_idx] * bundle.feature_std + bundle.feature_mean

print(
    f"Sample #{sample_idx}: {sample_meta['team_name']} vs {sample_meta['opp_team_name']} "
    f"(match {sample_meta['match_id']}, period {sample_meta['period_value']}, "
    f"ending at minute {sample_meta['minute']}) -> goal_next_{HORIZON_MINUTES}m = {int(bundle.train.y[sample_idx])}"
)
end_t = int(sample_meta["minute"])
minute_labels = list(range(end_t - SEQUENCE_LENGTH + 1, end_t + 1))
sample_df = pd.DataFrame(sample_raw, index=minute_labels, columns=bundle.feature_names)
sample_df.index.name = "minute"
display(sample_df.round(3))

## Krok 3 — Opakowanie zbiorów w PyTorch DataLoaders

`TensorDataset` łączy pary tensorów `(x, y)`, a `DataLoader` podaje je w paczkach (mini-batchach) po `BATCH_SIZE` próbek. Podczas treningu mieszamy kolejność paczek (`shuffle=True`), natomiast w `val` i `test` zachowujemy stałą kolejność.

In [ ]:
def make_loader(split, batch_size: int, shuffle: bool) -> DataLoader:
    """Converts a SequenceSplit (NumPy arrays) into a PyTorch DataLoader."""
    x_tensor = torch.tensor(split.x, dtype=torch.float32)
    y_tensor = torch.tensor(split.y, dtype=torch.float32)
    dataset = TensorDataset(x_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


train_loader = make_loader(bundle.train, BATCH_SIZE, shuffle=True)
val_loader = make_loader(bundle.val, BATCH_SIZE, shuffle=False)
test_loader = make_loader(bundle.test, BATCH_SIZE, shuffle=False)

# Sanity-check the shape of a single mini-batch.
batch_x, batch_y = next(iter(train_loader))
print("Mini-batch x shape:", tuple(batch_x.shape), "-> (batch_size, seq_len, num_features)")
print("Mini-batch y shape:", tuple(batch_y.shape), "-> (batch_size,)")

## Krok 4 — Definicja modelu LSTM

Jak `GoalPredictorLSTM` czyta 10-minutową sekwencję:
1. Warstwa `nn.LSTM` przechodzi po kolei przez minuty `t - 9, t - 8, ..., t`, aktualizując ukryty wektor pamięci o rozmiarze `HIDDEN_SIZE` (`32`).
2. Pobieramy ostatni stan ukryty `h_n[-1]` z minuty `t`, który podsumowuje całą 10-minutową historię.
3. `nn.Dropout` regularyzuje tę reprezentację, ograniczając przeuczenie.
4. `nn.Linear(HIDDEN_SIZE, 1)` zwraca pojedynczy surowy wynik (**logit**). Po przejściu przez `sigmoid(logit)` otrzymamy przewidywane prawdopodobieństwo gola.

In [ ]:
class GoalPredictorLSTM(nn.Module):
    """Simple, explainable LSTM binary classifier for next-M-minute goal prediction."""

    def __init__(
        self,
        num_features: int,
        hidden_size: int = 32,
        num_layers: int = 1,
        dropout: float = 0.25,
    ) -> None:
        super().__init__()
        # batch_first=True means input tensor shape is (batch, seq_len, num_features).
        self.lstm = nn.LSTM(
            input_size=num_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Run the sequence through the LSTM.
        # h_n shape: (num_layers, batch_size, hidden_size)
        _, (h_n, _) = self.lstm(x)

        # Take the last layer's final hidden state at minute t: (batch_size, hidden_size)
        last_hidden = h_n[-1]

        # Project to one logit per sample and remove the trailing size-1 dimension -> (batch_size,)
        logits = self.head(self.dropout(last_hidden))
        return logits.squeeze(-1)


model = GoalPredictorLSTM(
    num_features=len(bundle.feature_names),
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {num_params:,}")

## Krok 5 — Funkcja straty i metryki oceny

- **Niezbalansowanie klas**: gol pada tylko w ~14% 10-minutowych okien. `BCEWithLogitsLoss(pos_weight=neg/pos)` zwiększa wagę okien z golem w funkcji straty, dzięki czemu model uczy się odróżniać groźny napór od zwykłej gry.
- **Metryki**: zwykła trafność (accuracy) przy 14% przypadków pozytywnych jest myląca, dlatego mierzymy:
  - **ROC-AUC** (jakość rankingu: szansa, że okno przed golem dostanie wyższą ocenę niż okno bez gola; `0.500` = losowo).
  - **PR-AUC** (kompromis precyzji i czułości; poziom bazowy to odsetek goli ~`0.14`).
  - **Brier score** (błąd średniokwadratowy skalibrowanych prawdopodobieństw; im mniej, tym lepiej).
  - **Lift w Top 10%** (ile razy częściej pada gol w 10% najbardziej ryzykownych momentów względem średniej).

In [ ]:
pos_count = float((bundle.train.y == 1).sum())
neg_count = float((bundle.train.y == 0).sum())
pos_weight_value = neg_count / pos_count

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value], device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
print(f"Training positives: {int(pos_count):,} | negatives: {int(neg_count):,} | pos_weight: {pos_weight_value:.2f}")


def roc_auc_score_np(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Exact rank-based ROC-AUC using pure NumPy + SciPy rankdata (handles ties)."""
    n_pos = int((y_true == 1).sum())
    n_neg = int((y_true == 0).sum())
    if n_pos == 0 or n_neg == 0:
        return 0.5
    ranks = rankdata(y_score, method="average")
    return float((ranks[y_true == 1].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def pr_auc_score_np(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Average Precision (PR-AUC) sorted from highest predicted score to lowest."""
    n_pos = int((y_true == 1).sum())
    if n_pos == 0:
        return 0.0
    order = np.argsort(-y_score)
    y_sorted = y_true[order]
    tp = np.cumsum(y_sorted)
    precision = tp / np.arange(1, len(y_sorted) + 1)
    return float(np.sum(precision * y_sorted) / n_pos)


def brier_score_np(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    """Mean squared error between predicted probabilities and binary 0/1 outcomes."""
    return float(np.mean((y_prob - y_true) ** 2))


def top_k_lift(y_true: np.ndarray, y_prob: np.ndarray, fraction: float = 0.10) -> tuple[float, float]:
    """Returns (goal_rate_in_top_k, lift_over_base_rate) for the riskiest fraction of windows."""
    k = max(1, int(len(y_true) * fraction))
    top_idx = np.argsort(-y_prob)[:k]
    top_rate = float(y_true[top_idx].mean())
    base_rate = float(y_true.mean())
    return top_rate, top_rate / max(base_rate, 1e-8)

## Krok 6 — Pętla treningowa z wczesnym zatrzymaniem (Early Stopping)

W każdej epoce:
1. Przechodzimy cały zbiór `train_loader` (liczymy gradienty, przycinamy ich normę do `1.0` dla stabilności LSTM i aktualizujemy wagi przez `optimizer`).
2. Oceniamy model na `val_loader` bez liczenia gradientów (`torch.no_grad()`).
3. Zapisujemy kopię wag za każdym razem, gdy rośnie walidacyjne ROC-AUC; przerywamy trening, jeśli nie ma poprawy przez `PATIENCE` epok.

In [ ]:
def train_one_epoch(model: nn.Module, loader: DataLoader) -> float:
    """Runs one training epoch and returns mean weighted BCE loss."""
    model.train()
    total_loss = 0.0
    total_rows = 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += float(loss.item()) * len(y_batch)
        total_rows += len(y_batch)

    return total_loss / total_rows


@torch.no_grad()
def predict_logits(model: nn.Module, loader: DataLoader) -> tuple[float, np.ndarray, np.ndarray]:
    """Evaluates model without updating weights; returns (mean_loss, logits, y_true)."""
    model.eval()
    total_loss = 0.0
    total_rows = 0
    logits_list = []
    targets_list = []

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        logits = model(x_batch)
        loss = criterion(logits, y_batch)

        total_loss += float(loss.item()) * len(y_batch)
        total_rows += len(y_batch)
        logits_list.append(logits.cpu().numpy())
        targets_list.append(y_batch.cpu().numpy())

    return total_loss / total_rows, np.concatenate(logits_list), np.concatenate(targets_list)


best_val_auc = -1.0
best_state = None
epochs_without_improvement = 0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader)
    val_loss, val_logits, val_y = predict_logits(model, val_loader)
    val_auc = roc_auc_score_np(val_y, val_logits)
    val_pr = pr_auc_score_np(val_y, val_logits)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_auc": val_auc,
        "val_pr": val_pr,
    })

    marker = ""
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
        marker = " *best*"
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:2d}/{EPOCHS} | train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | val_ROC-AUC={val_auc:.4f} | val_PR-AUC={val_pr:.4f}{marker}"
    )
    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping triggered after {epoch} epochs (best val ROC-AUC: {best_val_auc:.4f}).")
        break

# Restore the best validation checkpoint before testing.
model.load_state_dict(best_state)
history_df = pd.DataFrame(history)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train loss")
ax1.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Val loss")
ax1.set_title("Weighted BCE Loss")
ax1.set_xlabel("Epoch")
ax1.legend()

ax2.plot(history_df["epoch"], history_df["val_auc"], marker="o", color="tab:green", label="Val ROC-AUC")
ax2.set_title("Validation ROC-AUC")
ax2.set_xlabel("Epoch")
ax2.legend()
plt.tight_layout()
plt.show()

## Krok 7 — Kalibracja Platta i ocena na zbiorze testowym

Ponieważ trenowaliśmy z `pos_weight`, surowe `sigmoid(logits)` zawyżają prawdopodobieństwa (zwykle krążą wokół ~`0.50` zamiast ~`0.14`). **Kalibracja Platta** dopasowuje dwa parametry `(a, b)` na zbiorze **walidacyjnym**, aby wzór `p = sigmoid(a * logit + b)` zwracał realistyczne prawdopodobieństwa na niewidzianych meczach **testowych** — bez podglądania etykiet testowych.

In [ ]:
def fit_platt_scaling(logits: np.ndarray, y_true: np.ndarray) -> tuple[float, float]:
    """Fits p = sigmoid(a * logit + b) by minimizing log-loss on the validation set."""
    def neg_log_likelihood(params: np.ndarray) -> float:
        probs = expit(params[0] * logits + params[1])
        return float(-np.mean(y_true * np.log(probs + 1e-9) + (1.0 - y_true) * np.log(1.0 - probs + 1e-9)))

    result = minimize(neg_log_likelihood, x0=np.array([1.0, 0.0]), method="Nelder-Mead")
    return float(result.x[0]), float(result.x[1])


_, val_logits, val_y = predict_logits(model, val_loader)
platt_a, platt_b = fit_platt_scaling(val_logits, val_y)

_, test_logits, test_y = predict_logits(model, test_loader)
test_probs = expit(platt_a * test_logits + platt_b)

# Constant baseline: always predict the training set's average goal frequency.
base_prob = float(bundle.train.y.mean())
baseline_probs = np.full_like(test_y, fill_value=base_prob)

test_roc = roc_auc_score_np(test_y, test_probs)
test_pr = pr_auc_score_np(test_y, test_probs)
test_brier = brier_score_np(test_y, test_probs)
base_brier = brier_score_np(test_y, baseline_probs)
top10_rate, top10_lift = top_k_lift(test_y, test_probs, fraction=0.10)

print(f"Platt calibration parameters (fitted on val): a = {platt_a:.3f}, b = {platt_b:+.3f}")
print(f"=== Test Set ({bundle.test.meta['match_id'].nunique()} unseen matches, {len(test_y):,} sequences) ===")
print(f"  Constant baseline : ROC-AUC = 0.5000 | PR-AUC = {test_y.mean():.4f} | Brier = {base_brier:.4f}")
print(f"  LSTM (20 features): ROC-AUC = {test_roc:.4f} | PR-AUC = {test_pr:.4f} | Brier = {test_brier:.4f}")
print(f"  Mean predicted P(goal): {test_probs.mean():.3%} (actual test goal rate: {test_y.mean():.3%})")
print(f"  Top 10% highest-risk moments: {top10_rate:.2%} goal rate ({top10_lift:.2f}x lift over average)")

## Krok 8 — Wizualizacja przebiegu dla jednego meczu testowego

Wybieramy jeden niewidziany w treningu mecz testowy z bramkami i rysujemy, jak przewidywane przez LSTM prawdopodobieństwo `P(gol w kolejnych M min)` zmienia się minuta po minucie dla **gospodarzy**, **gości** oraz **którejkolwiek drużyny** (`1 - (1 - p_home) * (1 - p_away)`).

In [ ]:
test_view = bundle.test.meta.copy()
test_view["p_goal"] = test_probs
test_view["y_actual"] = test_y

# Pick the test match with the most positive goal windows for an informative plot.
match_goal_counts = test_view.groupby("match_id")["y_actual"].sum()
selected_match_id = int(match_goal_counts.idxmax())
match_rows = test_view[test_view["match_id"] == selected_match_id].sort_values(["period_value", "minute"])

home_rows = match_rows[match_rows["is_home"] == 1.0].set_index("minute")
away_rows = match_rows[match_rows["is_home"] == 0.0].set_index("minute")
home_name = home_rows["team_name"].iloc[0]
away_name = away_rows["team_name"].iloc[0]

# Align home and away on common minutes to compute combined P(any goal in next M min).
common_mins = home_rows.index.intersection(away_rows.index)
p_home = home_rows.loc[common_mins, "p_goal"]
p_away = away_rows.loc[common_mins, "p_goal"]
p_any = 1.0 - (1.0 - p_home) * (1.0 - p_away)

# Detect minutes where live score_diff changed as goal markers.
score_changes = home_rows["score_diff"].diff().fillna(0.0)
home_goal_mins = score_changes[score_changes > 0].index.tolist()
away_goal_mins = score_changes[score_changes < 0].index.tolist()

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(common_mins, 100 * p_home, label=f"{home_name} (Home)", color="tab:blue", linewidth=2)
ax.plot(common_mins, 100 * p_away, label=f"{away_name} (Away)", color="tab:red", linewidth=2)
ax.plot(common_mins, 100 * p_any, label="Either team", color="black", linestyle="--", alpha=0.6)

for gm in home_goal_mins:
    ax.axvline(gm, color="tab:blue", linestyle=":", linewidth=1.8, alpha=0.8)
for gm in away_goal_mins:
    ax.axvline(gm, color="tab:red", linestyle=":", linewidth=1.8, alpha=0.8)

ax.set_title(
    f"Match {selected_match_id}: {home_name} vs {away_name} — "
    f"Predicted P(Goal in next {HORIZON_MINUTES} min)"
)
ax.set_xlabel("Expanded match minute (t)")
ax.set_ylabel("Predicted probability (%)")
ax.legend(loc="upper left")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Krok 9 — Zapis wytrenowanego modelu

Zapisujemy wagi sieci razem ze skalerem (`feature_mean`, `feature_std`), nazwami cech oraz parametrami kalibracji Platta, aby móc później oceniać nowe mecze bez ponownego trenowania.

In [ ]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "feature_names": list(bundle.feature_names),
    "feature_mean": bundle.feature_mean,
    "feature_std": bundle.feature_std,
    "sequence_length": bundle.sequence_length,
    "horizon_minutes": bundle.horizon_minutes,
    "hidden_size": HIDDEN_SIZE,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "platt_a": platt_a,
    "platt_b": platt_b,
}
torch.save(checkpoint, MODEL_PATH)
print(f"Saved LSTM checkpoint to: {MODEL_PATH}")

# Example: reloading later for inference
# ckpt = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
# loaded_model = GoalPredictorLSTM(len(ckpt["feature_names"]), ckpt["hidden_size"], ckpt["num_layers"], ckpt["dropout"])
# loaded_model.load_state_dict(ckpt["model_state_dict"])
# loaded_model.eval()